# 03 – DCF Valuation Model
**Author:** Mousumi Paul | Dec 2024

## 1. Setup

In [ ]:
import sys, pandas as pd, numpy as np
sys.path.append('../src')
from dcf_model import cost_of_equity, wacc, project_fcf, terminal_value, dcf_valuation, sensitivity_table
print('Setup complete ✅')

## 2. WACC Calculation

In [ ]:
# ── CAPM Inputs ──
risk_free_rate  = 0.072   # 10-yr G-Sec yield
beta            = 0.85
market_return   = 0.13

# ── Debt Inputs ──
cost_of_debt    = 0.075
tax_rate        = 0.25
equity_value    = 80000   # ₹ Crores (market cap)
debt_value      = 20000   # ₹ Crores

Ke = cost_of_equity(risk_free_rate, beta, market_return)
WACC = wacc(equity_value, debt_value, Ke, cost_of_debt, tax_rate)

print(f"Cost of Equity (CAPM): {Ke*100:.2f}%")
print(f"WACC: {WACC*100:.2f}%")

## 3. FCF Projection

In [ ]:
BASE_FCF     = 5000         # FY2023 FCF in ₹ Crores
GROWTH_RATES = [0.12, 0.11, 0.10, 0.09, 0.08]  # 5-year projections

fcfs = project_fcf(BASE_FCF, GROWTH_RATES)
for i, f in enumerate(fcfs, 1):
    print(f"  FY{2023+i}: ₹{f:,.1f} Cr")

## 4. Terminal Value & Intrinsic Value

In [ ]:
TGR      = 0.03    # Terminal Growth Rate
NET_DEBT = 2000    # ₹ Crores
SHARES   = 100     # Crores

tv     = terminal_value(fcfs[-1], TGR, WACC)
result = dcf_valuation(fcfs, tv, WACC, NET_DEBT, SHARES)

print("\n📊 DCF Valuation Results:")
for k, v in result.items():
    print(f"  {k}: ₹{v:,.2f}")

## 5. Sensitivity Analysis

In [ ]:
sens = sensitivity_table(
    BASE_FCF, GROWTH_RATES,
    wacc_range=[0.08, 0.09, 0.10, 0.11, 0.12],
    tgr_range=[0.02, 0.03, 0.04],
    net_debt=NET_DEBT,
    shares_outstanding=SHARES
)

print("\n📉 Sensitivity Table – Intrinsic Value Per Share (₹):")
print(sens.to_string())
sens.to_csv('../outputs/tables/dcf_sensitivity.csv')

## 6. Visualize Sensitivity Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(sens.astype(float), annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Intrinsic Value (₹)'})
ax.set_title('DCF Sensitivity: Intrinsic Value per Share\n(Rows = Terminal Growth Rate, Cols = WACC)')
plt.tight_layout()
plt.savefig('../outputs/charts/dcf_sensitivity_heatmap.png', dpi=150)
plt.show()